# nb45 - Physics-informed time features: reference-time fit and per-cell pulls (H9)

**Error analysis.** Inner regions R0/R1 sit at 0.070/0.061 vs 0.036 outside (nb37/nb38) and geometry conditioning is falsified - the deficit is pileup density, and per-cell time is the only unused physics handle there. Our models feed raw median-centered times; the timing lever plateaued (nb16 0.0665->0.0564, nb36 pair-dt flat).

**Question.** Does the SOTA time representation - an energy-weighted reference-time fit with per-cell compatibility pulls - extract more than raw median-centered times?

**Hypothesis.** H9: replacing raw centered times with TOF-corrected pulls (t - t0)/sigma_t(E), where t0 is a sigma_t-weighted mean with one 3-sigma rejection pass and sigma_t(E) is measured from clean data, improves sigma_eff in the E>17 GeV bins and/or R0/R1; a learnable pull-based soft time-gate multiplying the energy gate adds on top. Data check 2026-07-31: raw times are NOT TOF-corrected (per-event median ~86 ns, corr 0.85 with radius), and the within-window path spread is the same order as sigma_t - so the flight-path correction is a real, free sharpening of the pulls.

**Research.** CMS HGCAL time reconstruction: resolution-weighted mean restricted to compatible hits, sigma_t ~ 1/E (EPJ Web Conf 320 00046, 2025; arXiv:2005.13324). Belle II energy-dependent in-time gate width ~ const/E (arXiv:2203.11349). ATLAS DIPz per-object (mu, sigma) combination (ATL-DAQ-PUB-2026-002). PicoCal design timing JINST 21 C03006.

**Proof criterion.** Quant objective, pure-minbias training, 2 seeds per config; anchors = nb43 quant (0.0445 +/- 0.0001 singles, 2-seed ens 0.0437). Win = beat anchor by >0.002 overall, in any E>17 bin, or in R0/R1. sigma_t(E) is fit on clean data only (no test leakage).

This notebook imports the frozen pipeline from `scripts/` (picocal_data) per the 2026-07-30 mentor directive.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'plotly_white'
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
from picocal_data import build_grid, splits_for, THRESH, NC
OUT = REPO / 'reports' / 'predictions'
DEVICE = os.environ.get('NB45_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB45_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
print(f'device {DEVICE} | mode {MODE} | build {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events
device cuda | mode full | build 148s


## Time-of-flight correction (data-driven)

Measured on this dataset: per-event median cell time is ~86 ns and correlates 0.85 with radial distance - the raw times are NOT TOF-corrected. Within a 9x9 window the cell-to-cell path difference is O(0.1 ns), the same order as sigma_t, so we correct each cell by a straight-line-from-origin flight path (CMS HGCAL recipe) with an effective plane distance Z_eff fit from clean data. All quantities fit on clean only.

In [2]:
from scipy.optimize import curve_fit
C_MM_NS = 299.792458
W = 4
rp, rmp, dtp = [], [], []
for ev in CE:
    m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
    r = np.hypot(ev['x'][m], ev['y'][m])
    for tkey in ('tf', 'tb'):
        t = ev[tkey][m]
        v = np.isfinite(t)
        if v.sum() < 3: continue
        rm = float(np.median(r[v])); tm = float(np.median(t[v]))
        rp.append(r[v]); rmp.append(np.full(v.sum(), rm)); dtp.append(t[v] - tm)
rp, rmp, dtp = np.concatenate(rp), np.concatenate(rmp), np.concatenate(dtp)
def tof_resid(X, Z):
    r, rm = X
    return (np.sqrt(r ** 2 + Z ** 2) - np.sqrt(rm ** 2 + Z ** 2)) / C_MM_NS
(Z_EFF,), _ = curve_fit(tof_resid, (rp, rmp), dtp, p0=[12500.0], maxfev=10000)
Z_EFF = float(abs(Z_EFF))
def tof(r): return np.sqrt(r ** 2 + Z_EFF ** 2) / C_MM_NS
res_before = dtp
res_after = dtp - tof_resid((rp, rmp), Z_EFF)
def rsig(x):
    x = np.sort(x); n = len(x); k = max(1, int(0.683 * n))
    return 0.5 * np.min(x[k:] - x[:n - k])
print(f'Z_eff = {Z_EFF:.0f} mm | within-window time spread: before TOF {rsig(res_before):.3f} ns -> after {rsig(res_after):.3f} ns [{len(rp)} cell-times]')

Z_eff = 10746 mm | within-window time spread: before TOF 0.363 ns -> after 0.363 ns [1798840 cell-times]


## Measure sigma_t(E) from clean data (TOF-corrected)

Per clean window: event time = median of valid TOF-corrected cell times; residual per cell vs its energy; robust 68% width per log-energy bin; fit sigma_t(E) = sqrt((A/E)^2 + B^2). Fit on clean only - no leakage into the minbias test split.

In [3]:
def robust_sigma(x):
    x = np.sort(np.asarray(x))
    n = len(x)
    if n < 8: return np.nan
    k = max(1, int(np.floor(0.683 * n)))
    return 0.5 * np.min(x[k:] - x[:n - k])
pairs = []
for ev in CE:
    m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
    r = np.hypot(ev['x'][m], ev['y'][m])
    for tkey, ekey in (('tf', 'fr'), ('tb', 'bk')):
        t = ev[tkey][m] - tof(r); e = ev[ekey][m]
        v = np.isfinite(t)
        if v.sum() < 3: continue
        t0e = np.median(t[v])
        pairs.append(np.stack([e[v], t[v] - t0e], 1))
pairs = np.concatenate(pairs)
pairs = pairs[pairs[:, 0] > 0]
edges = np.quantile(pairs[:, 0], np.linspace(0, 1, 13))
ectr, svals = [], []
for i in range(12):
    hi = edges[i+1] + (1e-9 if i == 11 else 0)
    mm = (pairs[:, 0] >= edges[i]) & (pairs[:, 0] < hi)
    s = robust_sigma(pairs[mm, 1])
    if np.isfinite(s): ectr.append(float(np.median(pairs[mm, 0]))); svals.append(s)
ectr, svals = np.array(ectr), np.array(svals)
from scipy.optimize import curve_fit
def sigt_model(E, A, B): return np.sqrt((A / E) ** 2 + B ** 2)
(A_T, B_T), _ = curve_fit(sigt_model, ectr, svals, p0=[50.0, 0.2], maxfev=10000)
A_T, B_T = float(abs(A_T)), float(abs(B_T))
def sigt(E): return np.sqrt((A_T / np.maximum(E, 1e-3)) ** 2 + B_T ** 2)
print(f'sigma_t(E) = ({A_T:.1f} MeV*ns / E) (+) {B_T:.3f} ns   [{len(pairs)} clean cell-times]')
fig = go.Figure()
fig.add_scatter(x=ectr, y=svals, mode='markers', name='clean data')
xe = np.geomspace(ectr.min(), ectr.max(), 100)
fig.add_scatter(x=xe, y=sigt(xe), mode='lines', name='fit A/E (+) B')
fig.update_layout(height=380, xaxis_type='log', xaxis_title='cell energy [MeV]',
                  yaxis_title='sigma_t [ns]', title='per-cell time resolution from clean data')
fig.show()

sigma_t(E) = (1.4 MeV*ns / E) (+) 0.245 ns   [1798840 clean cell-times]


Windows with pull features: t0 = sigma_t-weighted mean over all timed cells (front+back), one 3-sigma rejection pass; token time columns become clipped pulls; pmin = per-cell best |pull| for the gate.

In [4]:
def make_windows_pull(W, EVS):
    rows = []; keep = []
    for i, ev in enumerate(EVS):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        rr = np.hypot(ev['x'][m], ev['y'][m])
        tf = tf - tof(rr); tb = tb - tof(rr)
        vf, vb = np.isfinite(tf), np.isfinite(tb)
        tv = np.concatenate([tf[vf], tb[vb]])
        sv = np.concatenate([sigt(fr[vf]), sigt(bk[vb])])
        if len(tv) >= 1:
            wgt = 1.0 / sv ** 2
            t0w = float((tv * wgt).sum() / wgt.sum())
            pl = np.abs(tv - t0w) / sv
            ok2 = pl < 3.0
            if ok2.sum() >= 1 and ok2.sum() < len(tv):
                t0w = float((tv[ok2] * (wgt[ok2])).sum() / wgt[ok2].sum())
        else:
            t0w = 0.0
        pf = np.where(vf, (tf - t0w) / sigt(np.maximum(fr, 1e-3)), 0.0)
        pb = np.where(vb, (tb - t0w) / sigt(np.maximum(bk, 1e-3)), 0.0)
        htf = vf.astype(np.float32); htb = vb.astype(np.float32)
        pabs_f = np.where(vf, np.abs(pf), np.inf); pabs_b = np.where(vb, np.abs(pb), np.inf)
        pmin = np.minimum(pabs_f, pabs_b)
        hast = np.isfinite(pmin).astype(np.float32)
        pmin = np.where(np.isfinite(pmin), pmin, 0.0)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), np.clip(pf, -5, 5), np.clip(pb, -5, 5)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'], ev['reg'],
                     np.clip(pmin, 0, 10).astype(np.float32), hast))
        keep.append(i)
    return rows, np.array(keep)
rows, keep = make_windows_pull(W, ME)
ktr, kva, kte = splits_for(keep, len(ME))
N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
NG = 5
y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
Et = np.array([r[3] for r in rows], np.float32)
REG = np.array([r[4] for r in rows], int)
sumE = np.array([r[1] for r in rows], np.float32)
X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
P = np.zeros((N, L), np.float32); HT = np.zeros((N, L), np.float32)
for i, (tok, se, sde, et, rg, pmin, hast) in enumerate(rows):
    n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
    e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
    P[i, :n] = pmin; HT[i, :n] = hast
    lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
    fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
    G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
         G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(Eraw).to(DEVICE), P=torch.from_numpy(P).to(DEVICE),
         HT=torch.from_numpy(HT).to(DEVICE))
print(f'W={W}: N {N}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}, IN_DIM {IN_DIM}')
print(f'timed-cell fraction: {HT[M.astype(bool)].mean():.3f} | pull median {np.median(P[(P>0)]):.2f}')

W=4: N 72554, tr/va/te 50787/10883/10884, IN_DIM 16
timed-cell fraction: 0.810 | pull median 1.36


In [5]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96)
class SubNetTP(nn.Module):
    def __init__(self, in_dim, la0, lb0, tgate):
        super().__init__()
        d = CFG['d']; self.tgate = tgate
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + 5, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
        self.kt = nn.Parameter(torch.tensor(3.0)); self.at = nn.Parameter(torch.tensor(1.0))
    def forward(self, x, m, g, ecell, p, ht):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        if self.tgate:
            wt = torch.sigmoid(self.at * (self.kt - p))
            w = w * (ht * wt + (1.0 - ht))
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        pool = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([pool, g], 1))
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
QS = torch.tensor([0.25, 0.5, 0.75], device=DEVICE)
def pinball(q, yb):
    d = yb - q
    return torch.maximum(QS * d, (QS - 1) * d).mean()
def wcalib(qv, qt, yva):
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    pe = np.empty(len(qt))
    for g in range(3):
        if (gv == g).sum() < 10 or (gt == g).sum() == 0:
            a, b2 = np.polyfit(qv[:, 1], yva, 1)
        else:
            a, b2 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
        pe[gt == g] = np.exp(a * qt[gt == g, 1] + b2)
    return pe
def train_eval(config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetTP(IN_DIM, la0, lb0, tgate=(config == 'pullgate')).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb45_{config}_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b], T['P'][b], T['HT'][b])
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(fwd(b).cpu().numpy())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256, False):
                s += pinball(fwd(b), T['Y'][b]).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad()
            pinball(fwd(b), T['Y'][b]).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    pe = wcalib(run(kva), run(kte), y[kva])
    extra = dict(kt=float(model.kt.detach().cpu()), at=float(model.at.detach().cpu())) if config == 'pullgate' else {}
    return float(resolution(pe, Et[kte])['sigma_eff']), pe, extra

In [6]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
JOBS = {'smoke': [('pull', 0), ('pullgate', 0)],
        'full': [(cfg, s) for cfg in ('pull', 'pullgate') for s in (0, 1)]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb45_time_pulls{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
for config, seed in JOBS:
    if (config, seed) in done: print('skip', config, seed); continue
    t1 = time.time()
    sig, pe, extra = train_eval(config, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb45_pred{TAG}_{config}_s{seed}.npy', pe)
    row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1),
               kt=round(extra.get('kt', float('nan')), 3), at=round(extra.get('at', float('nan')), 3))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s) {extra}', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

pull seed 0: sigma_eff 0.0455 (1198s) {}


pull seed 1: sigma_eff 0.0466 (902s) {}


pullgate seed 0: sigma_eff 0.0483 (2107s) {'kt': 3.4223246574401855, 'at': 0.11397315561771393}


pullgate seed 1: sigma_eff 0.0488 (1892s) {'kt': 3.4365735054016113, 'at': 0.13990329205989838}


  config  seed  sigma_eff  elapsed    kt    at
    pull     0     0.0455     1198   NaN   NaN
    pull     1     0.0466      902   NaN   NaN
pullgate     0     0.0483     2107 3.422 0.114
pullgate     1     0.0488     1892 3.437 0.140


## Verdict vs nb43 quant anchor

Win = beat 0.0445 singles / 0.0437 ens by >0.002 overall, in any E>17 bin, or in regions R0/R1. Same split, same objective, same training set - the only change is the time representation.

In [7]:
te_e = Et[kte]; te_r = REG[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
def perreg(pe):
    out = {}
    for r in sorted(set(te_r)):
        mm = te_r == r
        if mm.sum() >= 100: out[int(r)] = resolution(pe[mm], te_e[mm])['sigma_eff']
    return out
print('anchor nb43 quant: singles 0.0445 +/- 0.0001 | ens 0.0437 | targets 0.06/0.045/0.035/0.032/0.030/0.030')
anch = [np.load(OUT / f'nb43_pred_quant_s{s}.npy') for s in (0, 1) if (OUT / f'nb43_pred_quant_s{s}.npy').exists()]
if anch and len(anch[0]) == len(te_e):
    ea = np.stack(anch).mean(0)
    print(f'anchor ens per-bin: ' + ' / '.join(f'{b:.4f}' for b in perbin(ea)))
    print(f'anchor ens per-region: {perreg(ea)}')
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
for cfg in ('pull', 'pullgate'):
    preds = [np.load(OUT / f'nb45_pred{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb45_pred{TAG}_{cfg}_s{s}.npy').exists()]
    if not preds: continue
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'{cfg:9s} mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('          per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))
    print(f'          per-region {perreg(ens)}')

anchor nb43 quant: singles 0.0445 +/- 0.0001 | ens 0.0437 | targets 0.06/0.045/0.035/0.032/0.030/0.030
anchor ens per-bin: 0.0659 / 0.0479 / 0.0376 / 0.0362 / 0.0344 / 0.0387
anchor ens per-region: {0: 0.0687, 1: 0.0608, 2: 0.0391, 3: 0.0355, 4: 0.0393}
pull      mean 0.0461 +/- 0.0006 | ens 0.0449
          per-bin 0.0666 / 0.0514 / 0.0370 / 0.0385 / 0.0356 / 0.0395
          per-region {0: 0.0697, 1: 0.0641, 2: 0.0404, 3: 0.0359, 4: 0.0398}
pullgate  mean 0.0486 +/- 0.0003 | ens 0.0478
          per-bin 0.0691 / 0.0512 / 0.0386 / 0.0392 / 0.0392 / 0.0473
          per-region {0: 0.0836, 1: 0.0675, 2: 0.0413, 3: 0.0369, 4: 0.0413}
